In [ ]:
import os
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
seeds = [42, 67, 99, 70, 73]

datasets = {
    "EchoNext": {
        72475: "runs-echonext",
        32768: "runs-echonext-32k",
        16384: "runs-echonext-16k",
        8192: "runs-echonext-8k",
        4096: "runs-echonext-4k",
        2048: "runs-echonext-2k",
        1024: "runs-echonext-1k",
        512: "runs-echonext-512",
        256: "runs-echonext-256",
    },
    "MIMIC-IV-ECG (ED)": {
        78470: "runs-mimic",
        32768: "runs-mimic-32k",
        16384: "runs-mimic-16k",
        8192: "runs-mimic-8k",
        4096: "runs-mimic-4k",
        2048: "runs-mimic-2k",
        1024: "runs-mimic-1k",
        512: "runs-mimic-512",
        256: "runs-mimic-256",
    },
    "CODE-15%": {
        74112: "runs-code15",
        32768: "runs-code15-32k",
        16384: "runs-code15-16k",
        8192: "runs-code15-8k",
        4096: "runs-code15-4k",
        2048: "runs-code15-2k",
        1024: "runs-code15-1k",
        512: "runs-code15-512",
        256: "runs-code15-256",
    },
    "PTB-XL": {
        17418: "runs-ptbxl",
        8722: "runs-ptbxl-8k",
        4356: "runs-ptbxl-4k",
        2175: "runs-ptbxl-2k",
        1091: "runs-ptbxl-1k",
        547: "runs-ptbxl-512",
        273: "runs-ptbxl-256",
    },
    "CinC Georgia": {
        8192: "runs-cinc",
        4096: "runs-cinc-4k",
        2048: "runs-cinc-2k",
        1024: "runs-cinc-1k",
        512: "runs-cinc-512",
        256: "runs-cinc-256",
    },
    "ZZU pediatric ECG": {
        8658: "runs-zzu",
        4096: "runs-zzu-4k",
        2048: "runs-zzu-2k",
        1024: "runs-zzu-1k",
        512: "runs-zzu-512",
        256: "runs-zzu-256",
    },
}

# mapping of experiment name to tuple of:
# - plotting color
# - folder name
experiments = {
    "Blackbox Direct":     ("tab:grey",   "blackbox-direct"),
    "ECGFounder":          ("tab:red",    "ecgfounder-logreg"),
    "LabSup Proto Direct": ("tab:green",  "labsup-proto-direct"),
    "ProtoSSL HEEDB (PILA)":      ("tab:blue",   "protossl-heedb-pila"),
    "ProtoSSL HEEDB (PILA) (FT)": ("tab:cyan",   "protossl-heedb-pila-ft"),
    "LabSup Proto HEEDB (RILA)":      ("tab:orange", "labsup-proto-heedb-rila"),
    "LabSup Proto HEEDB (RILA) (FT)": ("tab:pink", "labsup-proto-heedb-rila-ft"),
}


def get_palette(exp_names):
    palette = dict()
    exps = experiments
    for exp_name in exp_names:
        palette[exp_name] = exps[exp_name][0]
    return palette

In [ ]:
data = []
for seed in seeds:
    output_dir = Path(f"/opt/gpu_working/steven/protossl-outputs-seed{seed}")
    for ds, sizes in datasets.items():
        for size, run_dir in sizes.items():
            exps = experiments.copy()
            for exp_name, (exp_color, exp_dir) in exps.items():
                metrics_csv = output_dir / run_dir / exp_dir / "metrics.csv"
                if not os.path.exists(metrics_csv):
                    continue
                metrics = pd.read_csv(metrics_csv, index_col="Label")
                multilabel = metrics.loc["Multilabel Averaged"]
                datum = {
                    "Seed": seed,
                    "Dataset": ds,
                    "Model": exp_name,
                    "Train Size": size,
                    "Multilabel (AUROC)": multilabel["AUROC"],
                    "Multilabel (AUPRC)": multilabel["AUPRC"],
                }
                data.append(datum)
results = pd.DataFrame.from_records(data)

In [ ]:
def plot_lift(
    *,  # enforce kwargs
    df: pd.DataFrame, # long format df (each point to plot is a row)
    dataset: str,
    seed: int | list[int] = 42, # if list of int, average across seeds
    metric: str,
    models: list[str], # must be intentional about which models to plot
    rename: list[str] | None = None,
    baseline_model: str | None = None, # singular result to optionally plot as dashed line
    ylim: tuple[float, float] | None = None,
    xlim: tuple[float, float] | None = None,
    save_path: str | None = None,
    smooth: bool = True,
    ax: plt.Axes | None = None,
):
    if rename is not None:
        assert len(models) == len(rename)
    if baseline_model is not None and baseline_model not in models:
        models = [baseline_model] + models
        if rename is not None:
            rename = [baseline_model] + rename
    palette = get_palette(models)
    if rename is not None:
        palette = {new_name: palette[m] for new_name, m in zip(rename, models)}
        df = df.copy()
        df["Model"] = df["Model"].replace({v: k for k, v in zip(rename, models)})

    df = df[df["Dataset"] == dataset]
    if isinstance(seed, int):
        df = df[df["Seed"] == seed]
    else: # list of seeds
        df = df[df["Seed"].isin(seed)]
        df = df.groupby(["Dataset", "Model", "Train Size"]).mean().reset_index()
    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    min_size = df["Train Size"].min()
    max_size = df["Train Size"].max()
    if xlim is not None:
        min_size = min(min_size, xlim[0])
        max_size = max(max_size, xlim[1])

    if baseline_model is not None:
        mask = df["Model"] == baseline_model
        assert (
            mask.sum() == 1
        ), f"Should only have 1 entry for baseline model: {baseline_model}"
        baseline_row = df[mask].iloc[0]
        ax.hlines(
            baseline_row[metric],
            min_size,
            max_size,
            colors=palette.pop(baseline_model),
            linestyles=":",
            label=baseline_model,
        )
        df = df[~mask] # subsequent line plots should exclude baseline model

    if not smooth:
        sns.lineplot(
            df,
            x="Train Size",
            y=metric,
            hue="Model",
            palette=palette,
            hue_order=list(palette.keys()),
            marker="o",
            ax=ax,
        )
    else:
        for model, color in palette.items():
            sns.regplot(
                df[df["Model"] == model],
                x="Train Size",
                y=metric,
                color=color,
                marker="o",
                ax=ax,
                logx=True,
                label=model,
            )
    ax.set_xscale("log", base=2)
    if xlim is not None:
        ax.set_xlim(xlim)
    else:
        # tighter boundaries than default lims
        ax.set_xlim((min_size, max_size))
    ax.set_title(f"{dataset} {metric}")
    ax.legend(loc="lower right")
    if ylim is not None:
        ax.set_ylim(ylim)
    if save_path is not None:
        assert fig is not None
        fig.tight_layout()
        fig.savefig(save_path)

In [ ]:
# average across seeds, linear probe top row, fine tune bottom row

fig, axs = plt.subplots(nrows=2, ncols=6, figsize=(36, 12))
for col in range(6):
    axs[1, col].sharey(axs[0, col])
for col, (ds, short) in enumerate([
    ("EchoNext", "echonext"),
    ("MIMIC-IV-ECG (ED)", "mimic"),
    ("PTB-XL", "ptbxl"),
    ("CinC Georgia", "cinc"),
    ("ZZU pediatric ECG", "zzu"),
    ("CODE-15%", "code15"),
]):
    plot_lift(
        df=results,
        dataset=ds,
        metric="Multilabel (AUROC)",
        models=[
            "Blackbox Direct",
            "LabSup Proto Direct",
            "LabSup Proto HEEDB (RILA)",
            "ProtoSSL HEEDB (PILA)",
        ],
        ax=axs[0, col],
        seed=seeds,
    )
    plot_lift(
        df=results,
        dataset=ds,
        metric="Multilabel (AUROC)",
        models=[
            "Blackbox Direct",
            "LabSup Proto Direct",
            "LabSup Proto HEEDB (RILA) (FT)",
            "ProtoSSL HEEDB (PILA) (FT)",
        ],
        ax=axs[1, col],
        seed=seeds,
    )
fig.tight_layout()
fig.savefig("figs/multiseed-avg.png")

In [ ]:
# plot each seed as a separate row

# ==============
# Linear Probe
# ==============

fig, axs = plt.subplots(nrows=5, ncols=6, figsize=(36, 30))
for col in range(6):
    for row in range(1, 5):
        axs[row, col].sharey(axs[0, col])
for row, seed in enumerate(seeds):
    for col, (ds, short) in enumerate([
        ("EchoNext", "echonext"),
        ("MIMIC-IV-ECG (ED)", "mimic"),
        ("PTB-XL", "ptbxl"),
        ("CinC Georgia", "cinc"),
        ("ZZU pediatric ECG", "zzu"),
        ("CODE-15%", "code15"),
    ]):
        plot_lift(
            df=results,
            dataset=ds,
            seed=seed,
            metric="Multilabel (AUROC)",
            models=[
                "Blackbox Direct",
                "LabSup Proto Direct",
                "LabSup Proto HEEDB (RILA)",
                "ProtoSSL HEEDB (PILA)",
            ],
            ax=axs[row, col],
        )
fig.tight_layout()
fig.savefig("figs/multiseed.png")

# ==============
# Fine-Tuned
# ==============

fig, axs = plt.subplots(nrows=5, ncols=6, figsize=(36, 30))
for col in range(6):
    for row in range(1, 5):
        axs[row, col].sharey(axs[0, col])
for row, seed in enumerate(seeds):
    for col, (ds, short) in enumerate([
        ("EchoNext", "echonext"),
        ("MIMIC-IV-ECG (ED)", "mimic"),
        ("PTB-XL", "ptbxl"),
        ("CinC Georgia", "cinc"),
        ("ZZU pediatric ECG", "zzu"),
        ("CODE-15%", "code15"),
    ]):
        plot_lift(
            df=results,
            dataset=ds,
            seed=seed,
            metric="Multilabel (AUROC)",
            models=[
                "Blackbox Direct",
                "LabSup Proto Direct",
                "LabSup Proto HEEDB (RILA) (FT)",
                "ProtoSSL HEEDB (PILA) (FT)",
            ],
            ax=axs[row, col],
        )
fig.tight_layout()
fig.savefig("figs/multiseed-ft.png")

In [ ]:
# individual files, single seed

# for ds, short in [
#     ("EchoNext", "echonext"),
#     ("MIMIC-IV-ECG (ED)", "mimic"),
#     ("PTB-XL", "ptbxl"),
#     ("CinC Georgia", "cinc"),
#     ("ZZU pediatric ECG", "zzu"),
#     ("CODE-15%", "code15"),
# ]:
#     plot_lift(
#         df=results,
#         dataset=ds,
#         metric="Multilabel (AUROC)",
#         models=[
#             "Blackbox Direct",
#             "LabSup Proto Direct",
#             "LabSup Proto HEEDB (RILA)",
#             "ProtoSSL HEEDB (PILA)",
#         ],
#         save_path=f"figs/probe-{short}.png",
#     )
#     plot_lift(
#         df=results,
#         dataset=ds,
#         metric="Multilabel (AUROC)",
#         models=[
#             "Blackbox Direct",
#             "LabSup Proto Direct",
#             "LabSup Proto HEEDB (RILA) (FT)",
#             "ProtoSSL HEEDB (PILA) (FT)",
#         ],
#         save_path=f"figs/ft-{short}.png",
#     )